# Coursework 2 - commented solution runner

This notebook is a Jupyter front end for the commented Python files in the lab root. The actual implementation stays in Python modules because the coursework assessor expects those module filenames and APIs. Run the sections in order.

## Before running

1. Meshcat connects to an existing server or starts one automatically.
2. Start Jupyter from the repository root with the `aro2026` Conda environment active.
3. Run the next cell to verify that this notebook is in the expected folder.

In [1]:
from pathlib import Path

project_root = Path.cwd()
assert (project_root / 'inverse_geometry.py').is_file(), (
    'Open this notebook from the lab repository root, not from another directory.'
)
print('Using project:', project_root)

Using project: /Users/amirfarisazra/VScode/MSC AI/ARO/lab


## 1. Read the inverse-geometry implementation

The solver aligns both end-effectors with the cube hooks using damped SE(3) inverse kinematics. It checks limits and collisions before accepting a configuration.

In [2]:
from IPython.display import Code, display
display(Code(filename='inverse_geometry.py', language='python'))

#!/usr/bin/env python3
"""Collision-checked bimanual inverse geometry for Coursework 2."""

import numpy as np
import pinocchio as pin

from config import EPSILON, LEFT_HAND, LEFT_HOOK, RIGHT_HAND, RIGHT_HOOK
from tools import collision, projecttojointlimits, setcubeplacement
from solution import remember_hand_relation


def _frame_placement(robot, q, frame_name):
    """Update kinematics and return the world placement of one named frame."""
    pin.framesForwardKinematics(robot.model, robot.data, q)
    return robot.data.oMf[robot.model.getFrameId(frame_name)]


def _cube_hook_target(cube, cube_placement, hook_name):
    """Return a hook's world pose without mutating its geometry placement."""
    pin.framesForwardKinematics(cube.model, cube.data, cube.q0)
    hook_id = cube.model.getFrameId(hook_name)
    return cube_placement * cube.data.oMf[hook_id]


def _pose_error_and_jacobian(robot, q, targets):
    """Stack the two 6-D local SE(3) errors and their matching Jacobians."""
    pin.computeJointJacobians(robot.model, robot.data, q)
    pin.framesForwardKinematics(robot.model, robot.data, q)

    errors, jacobians = [], []
    for frame_name, target in targets:
        frame_id = robot.model.getFrameId(frame_name)
        current = robot.data.oMf[frame_id]
        # log6(current^-1 target) is a 6-D body displacement taking the hand
        # from its current pose to the matching cube hook.
        errors.append(pin.log6(current.inverse() * target).vector)
        jacobians.append(
            pin.getFrameJacobian(robot.model, robot.data, frame_id, pin.ReferenceFrame.LOCAL)
        )
    return np.concatenate(errors), np.vstack(jacobians)


def _damped_least_squares(jacobian, error, damping):
    """Solve J dq = error robustly near a singular arm configuration."""
    jj_t = jacobian @ jacobian.T
    return jacobian.T @ np.linalg.solve(jj_t + damping * np.eye(jj_t.shape[0]), error)


def display_grasp(robot, cube, q, viz, cube_placement=None):
    """Draw the robot, carried cube and frame markers in one consistent pose.

    When no explicit cube pose is supplied, infer it from the left hand:
    world_T_cube = world_T_hand * inverse(cube_T_hook). This is geometric
    playback of an already planned grasp, not a physical Bullet simulation.
    """
    pin.framesForwardKinematics(robot.model, robot.data, q)
    if cube_placement is None:
        cube_to_hook = _cube_hook_target(cube, pin.SE3.Identity(), LEFT_HOOK)
        hand = robot.data.oMf[robot.model.getFrameId(LEFT_HAND)]
        cube_placement = hand * cube_to_hook.inverse()
    setcubeplacement(robot, cube, cube_placement)
    viz.display(q)
    for hand_name, hook_name in ((LEFT_HAND, LEFT_HOOK), (RIGHT_HAND, RIGHT_HOOK)):
        hand = robot.data.oMf[robot.model.getFrameId(hand_name)]
        viz.applyConfiguration(hand_name + "frame", hand)
        # Use the non-mutating helper: reading a hook must not move the cube.
        viz.applyConfiguration(
            hand_name + "HookFrame", _cube_hook_target(cube, cube_placement, hook_name)
        )


def _solve_grasp(robot, q_start, targets, posture, max_iterations, tolerance):
    """Run redundant, two-hand IK from one seed and return ``(q, success)``."""
    q = projecttojointlimits(robot, np.asarray(q_start, dtype=float).copy())
    posture = projecttojointlimits(robot, np.asarray(posture, dtype=float))
    damping = 1e-5

    for _ in range(max_iterations):
        error, jacobian = _pose_error_and_jacobian(robot, q, targets)
        if np.linalg.norm(error) < tolerance:
            return q, True

        primary_step = _damped_least_squares(jacobian, error, damping)
        # The null space leaves the hand task unchanged to first order.  It
        # biases the redundant torso/arms back to a comfortable configuration.
        null_space = np.eye(robot.model.nv) - np.linalg.pinv(jacobian) @ jacobian
        posture_step = null_space @ pin.difference(robot.model, q, posture)
        velocity = primary_step + 0.08 

In [3]:
from config import CUBE_PLACEMENT, CUBE_PLACEMENT_TARGET
from inverse_geometry import computeqgrasppose, display_grasp
from viewer import setupwithmeshcat

robot, cube, viz = setupwithmeshcat()
q0, start_ok = computeqgrasppose(robot, robot.q0, cube, CUBE_PLACEMENT, viz)
qe, goal_ok = computeqgrasppose(robot, q0, cube, CUBE_PLACEMENT_TARGET, viz)
print(f'Initial grasp found: {start_ok}; target grasp found: {goal_ok}')
if not (start_ok and goal_ok):
    raise RuntimeError('A grasp pose was not found; do not continue to planning.')
# Goal IK moved the cube; restore both robot and cube to the start.
display_grasp(robot, cube, q0, viz, CUBE_PLACEMENT)

pybullet build time: Oct 21 2025 17:04:55


Wrapper tries to connect to server <tcp://127.0.0.1:6000>
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7001/static/
Initial grasp found: True; target grasp found: True


## 2. Read and run the projected-RRT planner

The planner samples cube poses, then projects each edge through the IK solver. This keeps the grasp constraint at every displayed waypoint.

In [4]:
display(Code(filename='path.py', language='python'))

#!/usr/bin/env python3
"""Projected RRT planner for a rigid bimanual grasp."""

from dataclasses import dataclass
import time

import numpy as np
import pinocchio as pin

from inverse_geometry import computeqgrasppose, display_grasp, _cube_hook_target
from tools import collision, setcubeplacement
from tools import jointlimitsviolated
from config import LEFT_HAND, RIGHT_HAND, LEFT_HOOK, RIGHT_HOOK, EPSILON


@dataclass
class _Node:
    """One RRT node: its cube pose, grasping robot pose, and parent index."""

    cube_pose: pin.SE3
    q: np.ndarray
    parent: int | None


def _interpolate_pose(start, end, alpha):
    """Interpolate a full SE(3) cube pose rather than only its translation."""
    displacement = pin.log6(start.inverse() * end).vector
    return start * pin.exp6(displacement * alpha)


def _cube_distance(first, second):
    """Use metres plus a small rotation penalty as the RRT distance metric."""
    translation = np.linalg.norm(first.translation - second.translation)
    rotation = np.linalg.norm(pin.log3(first.rotation.T @ second.rotation))
    return translation + 0.10 * rotation


def _project_segment(robot, cube, q_start, cube_start, cube_end, step=0.025):
    """Project a cube-space edge onto valid grasping configurations.

    Each intermediate point is re-solved using the preceding solution as its
    seed.  Consequently every returned configuration satisfies the bimanual
    grasp constraint; ordinary joint-space linear interpolation would not.
    """
    distance = _cube_distance(cube_start, cube_end)
    samples = max(1, int(np.ceil(distance / step)))
    path, q_previous = [], q_start

    for index in range(1, samples + 1):
        target = _interpolate_pose(cube_start, cube_end, index / samples)
        q_next, success = computeqgrasppose(robot, q_previous, cube, target)
        if not success or collision(robot, q_next):
            return path, False
        path.append(q_next)
        q_previous = q_next
    return path, True


def _sample_cube_pose(rng, start, goal):
    """Sample the useful portion of the tabletop, with clearance around ends."""
    lo = np.minimum(start.translation, goal.translation) - np.array([0.20, 0.25, 0.0])
    hi = np.maximum(start.translation, goal.translation) + np.array([0.20, 0.25, 0.0])
    # Include clearance above the table: a height-only obstacle cannot be
    # traversed if every random sample is restricted to tabletop height.
    translation = np.array([
        rng.uniform(max(0.18, lo[0]), min(0.65, hi[0])),
        rng.uniform(max(-0.55, lo[1]), min(0.45, hi[1])),
        rng.uniform(min(start.translation[2], goal.translation[2]),
                    max(start.translation[2], goal.translation[2]) + 0.20),
    ])
    rotation = _interpolate_pose(start, goal, rng.uniform()).rotation
    return pin.SE3(rotation, translation)


def _finish_at_goal(robot, cube, path, qgoal, goal_pose):
    """Connect the IK branch to the requested exact endpoint, or reject it.

    Equivalent grasps can have different torso/elbow postures. For a nearby
    endpoint, explicitly check a finely sampled connecting segment for both
    hook alignment and collision; never silently return another configuration.
    """
    setcubeplacement(robot, cube, goal_pose)
    count = max(1, int(np.ceil(np.max(np.abs(qgoal - path[-1])) / 0.005)))
    tail = []
    targets = [(robot.model.getFrameId(hand), _cube_hook_target(cube, goal_pose, hook))
               for hand, hook in ((LEFT_HAND, LEFT_HOOK), (RIGHT_HAND, RIGHT_HOOK))]
    for alpha in np.linspace(0., 1., count + 1)[1:]:
        q = (1 - alpha) * path[-1] + alpha * qgoal
        pin.framesForwardKinematics(robot.model, robot.data, q)
        if jointlimitsviolated(robot, q) or collision(robot, q):
            return None
        if any(np.linalg.norm(pin.log6(robot.data.oMf[fid].inverse() * target).vector)
               > 5 * EPSILON for fid, target in targets):
            return None
        tail.append(q)
    tail[-1] = np.asarr

In [5]:
from path import computepath, displaypath

path = computepath(q0, qe, CUBE_PLACEMENT, CUBE_PLACEMENT_TARGET, robot, cube)
print(f'Collision-checked grasping waypoints: {len(path)}')
# Animate the cube alongside the hands at every waypoint.
displaypath(robot, path, dt=0.05, viz=viz, cube=cube)

Collision-checked grasping waypoints: 42


## 3. Read and run the dynamics controller

This cell runs the real PyBullet physics without a window. On macOS, opening Bullet inside a Jupyter kernel can leave the window unresponsive. To **watch the simulation**, open Terminal in this lab folder and run `../aro2026/bin/python control.py`. The controller follows the path with a speed-bounded quintic trajectory, inverse dynamics, and inward hand squeeze. The isolated simulator closes automatically at the end. See `README.md` for its model corrections.

In [6]:
display(Code(filename='control.py', language='python'))

#!/usr/bin/env python3
"""Time parametrisation and inverse-dynamics torque control for CW2."""

import numpy as np
import pinocchio as pin

from config import LEFT_HAND, LEFT_HOOK, RIGHT_HAND, RIGHT_HOOK


Kp = 300.0
Kv = 2.0 * np.sqrt(Kp)
SQUEEZE_FORCE = 8.0  # N per hand, maintained through physical contact
MAX_TORQUE = 180.0


class PiecewiseQuinticTrajectory:
    """A C2 trajectory that follows every configuration in a planned path.

    A fifth-order smooth-step has zero velocity and acceleration at both ends.
    Joining one curve per path edge therefore creates a C2 trajectory with a
    deliberate pause at each RRT waypoint, which is conservative but robust in
    the contact simulator.
    """

    def __init__(self, path, max_joint_speed=0.40):
        if len(path) < 2:
            raise ValueError("a trajectory needs at least two configurations")
        self.path = [np.asarray(q, dtype=float).copy() for q in path]
        changes = [np.max(np.abs(b - a)) for a, b in zip(self.path[:-1], self.path[1:])]
        if max_joint_speed <= 0:
            raise ValueError("max_joint_speed must be positive")
        # Quintic smoothstep reaches a peak slope of 1.875, not 1.
        self.durations = np.maximum(0.45, 1.875 * np.asarray(changes) / max_joint_speed)
        self.breakpoints = np.concatenate(([0.0], np.cumsum(self.durations)))
        self.duration = float(self.breakpoints[-1])

    def _segment(self, t):
        t = float(np.clip(t, 0.0, self.duration))
        index = min(np.searchsorted(self.breakpoints, t, side="right") - 1, len(self.durations) - 1)
        local_time = t - self.breakpoints[index]
        return index, np.clip(local_time / self.durations[index], 0.0, 1.0)

    def evaluate(self, t):
        """Return desired position, velocity and acceleration at time ``t``."""
        index, s = self._segment(t)
        delta = self.path[index + 1] - self.path[index]
        duration = self.durations[index]
        blend = 10 * s**3 - 15 * s**4 + 6 * s**5
        blend_dot = (30 * s**2 - 60 * s**3 + 30 * s**4) / duration
        blend_ddot = (60 * s - 180 * s**2 + 120 * s**3) / duration**2
        return (
            self.path[index] + blend * delta,
            blend_dot * delta,
            blend_ddot * delta,
        )


def _reference_at(trajs, tcurrent):
    """Accept this solution's trajectory or the starter tuple of Bezier curves."""
    if hasattr(trajs, "evaluate"):
        return trajs.evaluate(tcurrent)
    return tuple(trajectory(tcurrent) for trajectory in trajs)


def _world_cube_hook(cube, hook_name, cube_placement):
    """Use the static URDF hook offset with the cube's simulated world pose."""
    pin.framesForwardKinematics(cube.model, cube.data, cube.q0)
    hook_in_cube = cube.data.oMf[cube.model.getFrameId(hook_name)]
    return cube_placement * hook_in_cube


def _grasp_wrench_torque(sim, robot, cube, q, vq):
    """Squeeze opposing faces and support weight through hand joint torques.

    A spring aimed at the exact hook gives zero force when aligned, so it
    cannot maintain a friction grasp. A small inward force supplies preload.
    Forces are applied ONLY to robot joints; the cube moves through contacts.
    """
    position, quaternion = sim.p.getBasePositionAndOrientation(sim.cubeId)
    cube_placement = pin.XYZQUATToSE3(np.asarray([*position, *quaternion]))
    mass = sum(sim.p.getDynamicsInfo(sim.cubeId, i)[0]
               for i in range(-1, sim.p.getNumJoints(sim.cubeId)))
    pin.computeJointJacobians(robot.model, robot.data, q)
    pin.framesForwardKinematics(robot.model, robot.data, q)

    total = np.zeros(robot.model.nv)
    for hand_name, sign in ((LEFT_HAND, -1), (RIGHT_HAND, 1)):
        frame_id = robot.model.getFrameId(hand_name)
        jacobian = pin.getFrameJacobian(
            robot.model, robot.data, frame_id, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED
        )
        force = sign * SQUEEZE_FORCE * cube_placement.rotation[:, 1]
        force += np.array([0., 0.

In [9]:
from demo import run_demo

# Bullet physics runs headlessly here; use `python control.py` in Terminal
# if you want to watch the window on macOS.
result = run_demo(gui=False, path=path,
                  start=CUBE_PLACEMENT, goal=CUBE_PLACEMENT_TARGET)
print(result)
assert result['success'], 'The physical carry did not pass the checks.'

argv[0]=
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
base_linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
LARM_EFF_Linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
RARM_EFF_Linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/B